# Chapter 14 — The Art of Term Structure Models: Drift

教材 pp. 166–175；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from short_rate_calculator import *

无波动常数短率还原指数折现；Ho–Lee 匹配平坦初始曲线；pp.169–170。

In [2]:
assert_allclose(GaussianDrift(0,0).discount(.03,5),np.exp(-.15))
model=HoLee.from_forward_derivative(lambda t:0.,.01)
assert_allclose(model.discount(.03,5),np.exp(-.15))

教材p.184参数；风险中性均值、方差以及零期限价格。

In [3]:
model=Vasicek(.0165,.11,.0095)
assert_allclose(model.drift(0,.02),.001485)
assert_allclose(model.discount(.02,0),1)
rng=np.random.default_rng(42)
paths=model.simulate_exact(pd.DataFrame(rng.normal(size=(1,40000))),.02,1)
moments=model.moments(.02,1)
assert_allclose(paths.iloc[-1].mean(),moments['mean'],atol=.00015)
assert_allclose(paths.iloc[-1].var(),moments['variance'],rtol=.03)
print(moments)

mean        0.021473
variance    0.000089
dtype: float64


## Euler确定路径与债券远期的一阶导数关系；pp.166–175

In [4]:
model=GaussianDrift(.002,.01)
out=model.simulate(pd.DataFrame({'z':[0.,0.]}),.03,.5)
assert_allclose(out.z,[.03,.031,.032])
model=Vasicek(.2,.05,.01)
t=5.; h=1e-4
derivative=-(np.log(model.discount(.03,t+h))-np.log(model.discount(.03,t-h)))/(2*h)
assert_allclose(model.forward(.03,t),derivative,rtol=1e-8)